# LAB: Xây dựng web demo phân loại hoa Iris bằng Streamlit

**Sinh viên:** SE203499 — Trần Minh Huy  
**Mô hình:** Gaussian Naive Bayes. Notebook này nạp dữ liệu, đánh giá trên tập Test, huấn luyện mô hình cuối trên toàn bộ Iris, kiểm tra bốn ca theo đề và tạo ứng dụng Streamlit.

- [Mở ứng dụng Streamlit](https://se203499-iris-classification.streamlit.app/)
- [Mã nguồn GitHub](https://github.com/TranMinhHuy15/AIL_IRIS_Classification)


## 1. Cài thư viện và nạp dữ liệu

Chạy lần lượt các ô từ trên xuống dưới trong Google Colab.

In [ ]:
# Cài các thư viện dùng cho mô hình, bảng dữ liệu, biểu đồ và giao diện.
%pip install -q streamlit scikit-learn pandas numpy matplotlib pillow


In [ ]:
# Nạp thư viện và tập Iris có sẵn trong scikit-learn.
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names
print(f'Số mẫu: {X.shape[0]} | Số đặc trưng: {X.shape[1]}')
print('Ba loài:', ', '.join(target_names))


## 2. Xem dữ liệu Iris

In [ ]:
# Tạo bảng có tên loài tiếng Việt để kiểm tra dữ liệu đầu vào.
iris_df = pd.DataFrame(X, columns=feature_names)
iris_df['Loài hoa'] = [target_names[i].capitalize() for i in y]
display(iris_df.head())
display(iris_df['Loài hoa'].value_counts().rename_axis('Loài').to_frame('Số mẫu'))


## 3. Đánh giá trên tập Test theo tỉ lệ 80/20

In [ ]:
# Chia có phân tầng để giữ tỷ lệ ba loài; chỉ mô hình đánh giá dùng Train 80%.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
evaluation_model = GaussianNB()
evaluation_model.fit(X_train, y_train)
test_accuracy = accuracy_score(y_test, evaluation_model.predict(X_test))
print(f'Accuracy trên tập Test: {test_accuracy:.2%}')


## 4. Huấn luyện mô hình cuối trên toàn bộ Iris

In [ ]:
# Mô hình đưa vào ứng dụng được huấn luyện trên cả 150 mẫu theo đề bài.
model = GaussianNB()
model.fit(X, y)
print('Đã huấn luyện Gaussian Naive Bayes trên toàn bộ 150 mẫu.')

# Hàm dự đoán trả về tên loài và xác suất của cả ba lớp.
def predict_iris(features):
    input_array = np.asarray(features, dtype=float).reshape(1, -1)
    predicted_index = int(model.predict(input_array)[0])
    probabilities = model.predict_proba(input_array)[0]
    return target_names[predicted_index].capitalize(), probabilities


## 5. Kiểm thử bốn trường hợp trong đề

In [ ]:
# Ba ca điển hình và một ca gần ranh giới giữa Versicolor/Virginica.
test_cases = {
    'Setosa': [5.1, 3.5, 1.4, 0.2],
    'Versicolor': [6.0, 2.9, 4.5, 1.5],
    'Virginica': [6.5, 3.0, 5.8, 2.2],
    'Điểm giữa hai loài': [6.0, 2.7, 5.0, 1.7],
}
rows = []
for case_name, values in test_cases.items():
    predicted_species, probabilities = predict_iris(values)
    rows.append({
        'Trường hợp': case_name,
        'Đầu vào (SL, SW, PL, PW)': str(values),
        'Dự đoán': predicted_species,
        **{target_names[i].capitalize(): f'{probabilities[i]:.2%}' for i in range(3)},
    })
test_results = pd.DataFrame(rows)
display(test_results)


### Nhận xét ca ranh giới

Với `[6.0, 2.7, 5.0, 1.7]`, mô hình chọn **Virginica** vì xác suất 59.01% cao hơn **Versicolor** 40.99%. Hai xác suất cách nhau 18.02 điểm phần trăm, cho thấy ca này khó phân biệt hơn các ví dụ điển hình.


## 6. Tạo mã ứng dụng Streamlit

Ô kế tiếp ghi cùng mã `app.py` trong gói nộp vào thư mục làm việc của Colab. App dùng `@st.cache_resource` để chỉ tải và huấn luyện mô hình một lần trong mỗi phiên chạy.

In [ ]:
%%writefile /content/app.py
# Các thư viện dùng để xử lý dữ liệu, huấn luyện mô hình, vẽ biểu đồ và dựng giao diện.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

st.set_page_config(
    page_title="Demo Phân Loại Hoa Iris Dataset",
    page_icon="🌸",
    layout="wide",
)


@st.cache_resource(show_spinner="Đang tải dữ liệu và huấn luyện Gaussian Naive Bayes...")
def load_model_bundle():
    """Đánh giá holdout, sau đó huấn luyện mô hình cuối trên toàn bộ Iris."""
    # Tách tập 80/20 để báo cáo accuracy theo đúng yêu cầu bài lab.
    iris = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data,
        iris.target,
        test_size=0.2,
        random_state=42,
        stratify=iris.target,
    )
    evaluation_model = GaussianNB()
    evaluation_model.fit(X_train, y_train)
    accuracy = accuracy_score(y_test, evaluation_model.predict(X_test))

    # Huấn luyện mô hình dùng cho dự đoán cuối cùng trên toàn bộ 150 mẫu.
    model = GaussianNB()
    model.fit(iris.data, iris.target)
    return iris, model, float(accuracy)


@st.cache_data
def load_iris_frame():
    # Tạo bảng dữ liệu có tên loài để hiển thị trong tab Dữ liệu Iris.
    iris = load_iris()
    frame = pd.DataFrame(iris.data, columns=iris.feature_names)
    frame["Loài hoa"] = [iris.target_names[target].capitalize() for target in iris.target]
    return frame


def predict_iris(model, values):
    # Chuyển bốn giá trị đầu vào thành đúng định dạng mà scikit-learn yêu cầu.
    input_data = np.asarray(values, dtype=float).reshape(1, -1)
    predicted_class = int(model.predict(input_data)[0])
    probabilities = model.predict_proba(input_data)[0]
    return predicted_class, probabilities


def show_probability_bars(target_names, probabilities):
    # Hiển thị xác suất của cả ba lớp dưới dạng thanh tiến trình.
    st.markdown("#### Xác suất từng loài")
    for index, probability in enumerate(probabilities):
        st.write(f"**{target_names[index].capitalize()}** — {probability:.2%}")
        st.progress(float(probability))


def show_petal_chart(frame, features, target_names):
    # Vẽ dữ liệu Petal Length/Petal Width và đánh dấu điểm người dùng nhập.
    fig, ax = plt.subplots(figsize=(8, 5))
    palette = ["#2E7D32", "#1976D2", "#D84315"]
    for index, species in enumerate(target_names):
        subset = frame[frame["target"] == index]
        ax.scatter(
            subset["petal length (cm)"],
            subset["petal width (cm)"],
            label=species.capitalize(),
            color=palette[index],
            alpha=0.65,
            s=38,
        )
    ax.scatter(
        features[2],
        features[3],
        marker="*",
        s=280,
        color="#F9A825",
        edgecolors="#222222",
        linewidths=1,
        label="Điểm bạn vừa nhập",
        zorder=5,
    )
    ax.set_title("Phân bố kích thước cánh hoa")
    ax.set_xlabel("Chiều dài cánh hoa (Petal Length - cm)")
    ax.set_ylabel("Chiều rộng cánh hoa (Petal Width - cm)")
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    st.pyplot(fig, clear_figure=True)
    plt.close(fig)


# Tải dữ liệu và mô hình đã cache; thao tác với slider không huấn luyện lại mô hình.
iris, model, test_accuracy = load_model_bundle()
iris_frame = load_iris_frame()
plot_frame = iris_frame.copy()
plot_frame["target"] = iris.target

# Sidebar tóm tắt thuật toán, dữ liệu và kết quả đánh giá trên tập Test.
with st.sidebar:
    st.header("Thông tin mô hình")
    st.write("**Thuật toán:** Gaussian Naive Bayes")
    st.write("**Dữ liệu:** Iris, 150 mẫu, 4 đặc trưng")
    st.write("**Huấn luyện:** toàn bộ dữ liệu Iris")
    st.metric("Accuracy trên tập Test", f"{test_accuracy:.2%}")
    st.caption("Tập Test: 20%, random_state=42, stratify=y.")

st.title("Demo Phân Loại Hoa Iris Dataset")
st.subheader("Mô hình: Gaussian Naive Bayes")
st.write("Điều chỉnh bốn thông số hoa, sau đó chọn **Dự đoán ngay** để xem kết quả.")

# Tab chính có bố cục hai cột; tab còn lại trình bày bảng dữ liệu gốc.
demo_tab, data_tab = st.tabs(["Dự đoán", "Dữ liệu Iris"])

with demo_tab:
    input_column, result_column = st.columns(2, gap="large")
    with input_column:
        st.markdown("### Thông số đầu vào")
        with st.form("iris_prediction_form"):
            sepal_length = st.slider(
                "Chiều dài lá đài (Sepal Length - cm)",
                min_value=4.0, max_value=8.0, value=5.1, step=0.1,
            )
            sepal_width = st.slider(
                "Chiều rộng lá đài (Sepal Width - cm)",
                min_value=2.0, max_value=4.5, value=3.5, step=0.1,
            )
            petal_length = st.slider(
                "Chiều dài cánh hoa (Petal Length - cm)",
                min_value=1.0, max_value=7.0, value=1.4, step=0.1,
            )
            petal_width = st.slider(
                "Chiều rộng cánh hoa (Petal Width - cm)",
                min_value=0.1, max_value=2.5, value=0.2, step=0.1,
            )
            submitted = st.form_submit_button("Dự đoán ngay", width="stretch")

        if submitted:
            # Ghi nhớ bộ thông số đã gửi để các lần chạy lại vẫn giữ kết quả.
            st.session_state["iris_last_input"] = [
                sepal_length, sepal_width, petal_length, petal_width,
            ]

    with result_column:
        st.markdown("### Kết quả")
        last_input = st.session_state.get("iris_last_input")
        if last_input is None:
            st.info("Kết quả sẽ hiển thị sau khi bạn bấm **Dự đoán ngay**.")
        else:
            # Chỉ dự đoán và trình bày kết quả sau khi người dùng bấm nút.
            predicted_class, probabilities = predict_iris(model, last_input)
            predicted_species = iris.target_names[predicted_class]
            st.success(f"Loài hoa được dự đoán: **{predicted_species.upper()}**")
            show_probability_bars(iris.target_names, probabilities)

            # Nếu thiếu ảnh, hiện cảnh báo để ứng dụng tiếp tục chạy bình thường.
            image_path = Path(__file__).resolve().parent / "assets" / f"{predicted_species}.jpg"
            if image_path.is_file():
                st.image(
                    str(image_path),
                    caption=f"Hoa Iris {predicted_species.capitalize()}",
                    width="stretch",
                )
            else:
                st.warning(f"Chưa có ảnh minh họa cho loài {predicted_species.capitalize()}.")

            with st.expander("Xem điểm dự đoán trên biểu đồ phân bố"):
                show_petal_chart(plot_frame, last_input, iris.target_names)

with data_tab:
    st.markdown("### Bảng dữ liệu Iris")
    st.caption("Dữ liệu gồm 150 mẫu, 4 đặc trưng đo kích thước và nhãn loài hoa.")
    st.dataframe(iris_frame, width="stretch", hide_index=True)


In [ ]:
# Tải ba ảnh minh họa từ repository công khai vào đúng thư mục mà app.py sử dụng.
from pathlib import Path
from urllib.request import urlretrieve

assets_dir = Path('/content/assets')
assets_dir.mkdir(parents=True, exist_ok=True)
for species in ['setosa', 'versicolor', 'virginica']:
    image_url = (
        'https://raw.githubusercontent.com/TranMinhHuy15/'
        f'AIL_IRIS_Classification/main/assets/{species}.jpg'
    )
    urlretrieve(image_url, assets_dir / f'{species}.jpg')
print('Đã tải đủ ba ảnh minh họa vào /content/assets.')


## 7. Chạy app ngay trong Colab

Chạy ô bên dưới để mở cổng 8501 và in link xem app. Colab tạo link proxy tạm thời cho phiên runtime; link Streamlit Community Cloud ở đầu notebook là link công khai ổn định để giảng viên xem app. Khi cần dừng app, khởi động lại runtime.

### Ảnh kiểm thử

Mở link Streamlit, nhập lần lượt bốn bộ giá trị ở phần kiểm thử, bấm **Dự đoán ngay**, rồi chụp vùng giao diện có thông số và kết quả để đính kèm báo cáo.

In [ ]:
# Khởi chạy Streamlit ở chế độ nền và lấy URL proxy của Google Colab.
import subprocess
import time
import socket
from google.colab import output

# Chọn cổng trống để chạy được cả khi runtime còn app cũ ở cổng 8501.
with socket.socket() as port_socket:
    port_socket.bind(('', 0))
    app_port = port_socket.getsockname()[1]
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', '/content/app.py', f'--server.port={app_port}', '--server.headless=true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)
time.sleep(5)
colab_app_url = output.eval_js(f'google.colab.kernel.proxyPort({app_port})')
print('Link app trong phiên Colab:', colab_app_url)
